# Purdue University Affiliation Analysis

This notebook analyzes awardee names for Purdue University connections using the Institution Checker pipeline.

**Workflow:**
1. **Setup**: Configure environment, reload modules, and set API keys.
2. **Data Loading**: Load and prepare the dataset (Full or Sample).
3. **Clustering**: Group similar names to optimize processing.
4. **Validation**: Run a quick random smoke test for sanity.
5. **Execution**: Run the full pipeline on the clustered dataset.
6. **Analysis**: Review results, optionally run unbiased recovery on expected-missed names, and export data.

In [1]:
# 1. SETUP & CONFIGURATION
# -----------------------------------------------------------------------------
# This cell handles module reloading and API configuration.
# Run this FIRST to ensure the environment is ready.

import sys
import importlib
import pandas as pd
import time
from pathlib import Path

# --- Fresh-run state reset (avoid stale notebook variables) ---
CURRENT_RUN_ID = None
RUN_RESULTS_READY = False
for var_name in [
    "df", "df_base", "df_vips", "df_others", "df_random", "df_clustered",
    "cluster_reps", "cluster_results", "cluster_institution_map", "final_results"
]:
    if var_name in globals():
        del globals()[var_name]

# --- Force Local Source Path (prevents stale UNC/site-package imports) ---
workspace_src = Path(r"e:\Awards DB Code\step3_institution_checker\src")
if workspace_src.exists():
    sys.path = [p for p in sys.path if "institution_checker" not in p.lower()]
    sys.path.insert(0, str(workspace_src))
    print(f"Using local source path: {workspace_src}")

# Drop cached institution_checker modules so imports resolve from the path above.
for mod in [m for m in list(sys.modules) if m == "institution_checker" or m.startswith("institution_checker.")]:
    del sys.modules[mod]

# --- Module Reloading (Development Mode) ---
# Reinstall local package to ensure latest changes are applied
!pip uninstall -y institution_checker
!pip install -e .

# Force reload of modules to pick up changes
modules_to_reload = [
    'institution_checker.config',
    'institution_checker.search',
    'institution_checker.llm_processor',
    'institution_checker.main',
    'institution_checker'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded {module_name}")

# --- Imports (AFTER Reloading) ---
# Import AFTER reloading to ensure we get the new function objects
import institution_checker
import institution_checker.llm_processor
import institution_checker.search
from institution_checker import (
    run_pipeline,
    resolve_names,
    set_api_key,
    expand_cluster_results_identity_safe,
    run_targeted_recovery_pass,
 )
from institution_checker.llm_processor import enable_prompt_logging
from name_matcher.pipeline import NameMatcher, NameMatcherConfig

print(f"institution_checker loaded from: {institution_checker.__file__}")

# --- API Configuration ---
API_KEY = "sk-3f1dbbf2450e46ab9541dffba4f18ec6"
set_api_key(API_KEY)

# Configure NameMatcher
config = NameMatcherConfig(name_column="name", use_tqdm=True)
config.llm_token = API_KEY

print("\nSetup complete:")
print("   - Fresh in-memory state initialized")
print("   - Modules reloaded and installed")
print("   - API keys configured")
print("   - Browser timeout fixes active (15s)")
print("   - Search strategies optimized (DuckDuckGo prioritized + Rate Limit Protection)")
print("   - Evidence gathering expanded (8 excerpts)")

Using local source path: e:\Awards DB Code\step3_institution_checker\src
Found existing installation: institution-checker 0.1.0
Uninstalling institution-checker-0.1.0:
  Successfully uninstalled institution-checker-0.1.0
Obtaining file:///E:/Awards%20DB%20Code/step3_institution_checker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for institution-checker (pyproject.toml): started
  Building editable for institution-checker (pyproject.toml): finished with status 'done'
  Created wheel for institution-checker: filename=institu

In [2]:
# 2. DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
# Select the dataset to process and prepare it for analysis.

# --- Configuration ---
DATASET_OPTION = "purdue"  # Options: "purdue", "nobel", "sample", "accuracy_test_25"

# File Paths
PATHS = {
    "purdue": r"E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx",
    "nobel": r"E:\Ace\Nobel Only Test Set.xlsx"
}

# --- Load Data ---
if DATASET_OPTION == "sample":
    print("ðŸ“‰ CREATING TEST SAMPLE: 9 VIPs + 100 Random Names")
    # Load base dataset (Nobel) for sampling
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].copy()
    df_others = df_base[~vip_mask].copy()
    
    sample_size = min(100, len(df_others))
    df_random = df_others.sample(n=sample_size, random_state=42)
    
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    print(f"âœ… Sample created: {len(df)} records")

elif DATASET_OPTION == "accuracy_test_25":
    print("ðŸ“‰ CREATING ACCURACY TEST: 9 VIPs + 16 Random Non-Connected")
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    # 1. Get VIPs
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].drop_duplicates(subset=['name']).copy()
    
    # 2. Get Others (Non-VIPs)
    df_others = df_base[~vip_mask].copy()
    
    # 3. Select 16 random names
    # Using a fixed seed ensures reproducibility. 
    df_random = df_others.sample(n=16, random_state=123) 
    
    # Combine
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    
    print(f"âœ… Test set created: {len(df)} records")
    print(f"   VIPs (Positive Controls): {len(df_vips)}")
    print(f"   Others (Negative Controls): {len(df_random)}")
    
    print("\nðŸ“‹ List of names to check:")
    print("-" * 30)
    for i, name in enumerate(df['name'], 1):
        role = "VIP" if any(v in name for v in vip_names) else "Other"
        print(f"{i:2}. {name:<30} [{role}]")

elif DATASET_OPTION in PATHS:
    path = PATHS[DATASET_OPTION]
    df = pd.read_excel(path, dtype=str)
    print(f"âœ… Loaded {len(df):,} records from {DATASET_OPTION.upper()} dataset")
    print(f"   Source: {path}")

else:
    raise ValueError(f"Invalid DATASET_OPTION: {DATASET_OPTION}")

# --- Clustering ---
print("\nðŸ§© Clustering similar names...")
matcher = NameMatcher(config)
result = matcher.cluster(df)
df_clustered = result.dataframe
print(f"âœ… Clustering complete: {len(df)} names -> {df_clustered['cluster_label'].nunique()} clusters")


âœ… Loaded 607 records from PURDUE dataset
   Source: E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx

ðŸ§© Clustering similar names...
--- Name Matcher Process Started (v6.8.8 Improved Canonical Name Selection) ---

1. Loading and preparing data...
   Loaded 607 names. Done in 0.01s
2. Vectorizing names for similarity scoring...
   Done in 0.01s
3. Generating candidate pairs via Blocking...
   Generated 682 candidate pairs from blocking.
   Done in 0.00s
4. Applying Stricter Cluster-Centric Filtering...


   Filtering Pairs: 100%|██████████| 682/682 [00:00<00:00, 340780.95pair/s]


   Filter Stats: {'merged_expansion': 299, 'rejected_transitive_conflict': 45}
   Done in 0.00s
5. Reviewing ambiguous pairs with LLM...
   Sending 6 unique pairs to LLM for review using 5 parallel workers...


   LLM Review: 100%|██████████| 1/1 [00:03<00:00,  3.70s/batch]

   Done in 3.71s
6. Finalizing cluster data structures...
   Done in 0.00s
7. Refining clusters by ejecting outliers...
   Found and ejected 0 outliers from clusters.
   Done in 0.09s
8. Assigning canonical names and saving results...

--- Results Summary ---
   - Total names processed: 607
   - Unique entities (clusters) found: 305

   --- Sample of Largest Clusters Found ---
   Cluster 1 (Size: 10): 'Seymour Benzer'
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - ...
   Cluster 2 (Size: 8): 'Alvin Carl Plantinga'
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - ...
   Cluster 3 (Size: 7): 'George Andrew Olah'
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - ...
   Cluster 4 (Size: 7): 'Hans Albrecht Bethe'
     - H. A. Bethe
     - H. A. Bethe
     - Hans A. Beth

In [3]:
# 3. PIPELINE VALIDATION (RANDOM SMOKE TEST)
# -----------------------------------------------------------------------------
# Run a quick random sanity check before full execution.

SMOKE_TEST_SIZE = 9

available_names = df['name'].dropna().astype(str).str.strip()
available_names = available_names[available_names != '']

if available_names.empty:
    raise RuntimeError('No names available for smoke test.')

sample_n = min(SMOKE_TEST_SIZE, len(available_names))
smoke_test_names = available_names.sample(n=sample_n, random_state=42).tolist()

print(f"Random smoke test ({sample_n} names)")
print("=" * 60)
print("Testing pipeline behavior on unbiased random names...")

start_time = time.time()

smoke_results = await run_pipeline(
    smoke_test_names,
    batch_size=3,
    use_enhanced_search=True,
    dataset_profile="high_connection",
    use_dynamic_batching=False
)

elapsed = time.time() - start_time

print(f"\nSMOKE TEST RESULTS ({elapsed:.1f}s, {elapsed/sample_n:.1f}s per name):")
print("-" * 60)

connected_count = 0
for name, res in zip(smoke_test_names, smoke_results):
    verdict = str(res.get('verdict', 'unknown')).lower()
    institution = str(res.get('institution', ''))
    is_purdue = 'purdue' in institution.lower()
    is_connected = verdict == 'connected' and is_purdue

    if is_connected:
        connected_count += 1
        icon = "✅"
    else:
        icon = "❌"

    print(f"   {icon} {name}: {verdict}")

print(f"\nConnected-to-Purdue in random smoke test: {connected_count}/{sample_n}")
print("Proceed to Section 4 for the full clustered run.")

Random smoke test (9 names)
Testing pipeline behavior on unbiased random names...
[PIPELINE] Starting: 9 name(s) in 3 batch(es) using enhanced search
[PIPELINE] Batch size: 3, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/3 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/3 =====
[PIPELINE] Names in this batch: ['Vernon W. Ruttan', 'Jay Hopler', 'Chintamani Nagesa Ramachandra Rao']
[BATCH] Processing 3 names: Vernon W. Ruttan, Jay Hopler, Chintamani Nagesa Ramachandra Rao
[BATCH] Phase 1: Running searches in parallel for all 3 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Jay Hopler: 14 results (mode=basic_only, queries=1, bucket=plausible, t_basic=7.7s, t_rescue=0.0s, t_enh=0.0s, t_total=7.7s)
[PROGRESS] Starting LLM analysis for: Jay Hopler
[BATCH] Search succeeded for Chintamani Nagesa Ramachandra Rao: 20 results (mode=basic_only, queries=1, bucket=plausible, t_basic=8.3s, t_rescue=0.0s, t_enh=0.0s, t_total=8.3s)
[PROGRESS] Starting LLM analysis for: Chintamani Nagesa Ramachandra Rao
[PROGRESS] LLM analysis completed for Chintamani Nagesa Ramachandra Rao in 4.1s
[PROGRESS] LLM analysis completed for Jay Hopler in 5.0s
[BATCH] Search succeeded

In [4]:
# 4. MAIN EXECUTION (FULL DATASET)
# -----------------------------------------------------------------------------
# Run the affiliation checker on the full clustered dataset using the optimized library pipeline.

TEST_MODE = False

# Profile selection by dataset intent:
# - purdue/sample/accuracy_test_25: higher expected positives -> prioritize recall
# - nobel: sparse positives -> optimize throughput
if DATASET_OPTION in {"purdue", "sample", "accuracy_test_25"}:
    DATASET_PROFILE = "high_connection"
    USE_DYNAMIC_BATCHING = False
    USE_ENHANCED_SEARCH = True
    BATCH_SIZE = 12
else:
    DATASET_PROFILE = "low_connection"
    USE_DYNAMIC_BATCHING = True
    USE_ENHANCED_SEARCH = False
    BATCH_SIZE = 30

CURRENT_RUN_ID = int(time.time())
RUN_RESULTS_READY = False

cluster_reps = df_clustered.groupby('cluster_label')['name'].first().reset_index()
cluster_reps.columns = ['cluster_label', 'representative_name']

if TEST_MODE:
    cluster_reps = cluster_reps.head(20)
    print(f"TEST MODE: Checking {len(cluster_reps)} clusters")
else:
    print(f"Checking all {len(cluster_reps)} clusters")

names_to_check = cluster_reps['representative_name'].tolist()

print("Starting Optimized Institution Algorithm (internal library v2)...")
print(f"   Dataset option: {DATASET_OPTION}")
print(f"   Profile: {DATASET_PROFILE}")
print(f"   Search Batch: {BATCH_SIZE}")
print(f"   Dynamic batching: {USE_DYNAMIC_BATCHING}")
print(f"   Enhanced search: {USE_ENHANCED_SEARCH}")
print(f"   Names to process: {len(names_to_check)}")
print(f"   Run ID: {CURRENT_RUN_ID}")
print("-" * 60)

start_time = time.time()
cluster_results = await run_pipeline(
    names_to_check,
    batch_size=BATCH_SIZE,
    use_enhanced_search=USE_ENHANCED_SEARCH,
    dataset_profile=DATASET_PROFILE,
    use_dynamic_batching=USE_DYNAMIC_BATCHING,
)

elapsed = time.time() - start_time
avg_per_name = elapsed / len(names_to_check) if names_to_check else 0
print(f"\nTIMING: total={elapsed:.1f}s | avg={avg_per_name:.2f}s/name")
print(f"THROUGHPUT: {60/avg_per_name:.1f} names/min" if avg_per_name > 0 else "THROUGHPUT: N/A")

fields_to_map = [
    'institution', 'verdict', 'relationship_type', 'relationship_timeframe',
    'verification_detail', 'summary', 'primary_source', 'verification_status',
    'temporal_context', 'confidence', 'purdue_connection',
    'pre_llm_bucket', 'pre_llm_stage', 'pre_llm_score', 'pre_llm_summary',
    'pre_llm_reason_codes', 'pre_llm_used_rescue_query'
]

df_clustered, cluster_institution_map = await expand_cluster_results_identity_safe(
    df_clustered,
    cluster_reps,
    cluster_results,
    batch_size=BATCH_SIZE,
    dataset_profile=DATASET_PROFILE,
    fields_to_map=fields_to_map,
    debug=False,
)

final_results = df_clustered.copy()
final_results['_run_id'] = CURRENT_RUN_ID
final_results['_run_generated_at'] = pd.Timestamp.utcnow().isoformat()
RUN_RESULTS_READY = True

if 'pre_llm_stage' in final_results.columns:
    print("Pre-LLM stage distribution:")
    print(final_results['pre_llm_stage'].fillna('(missing)').value_counts().head(10))

print("Affiliation check complete")
print(f"Fresh rows: {len(final_results)}")

Checking all 305 clusters
Starting Optimized Institution Algorithm (internal library v2)...
   Dataset option: purdue
   Profile: high_connection
   Search Batch: 12
   Dynamic batching: False
   Enhanced search: True
   Names to process: 305
   Run ID: 1775669329
------------------------------------------------------------
[PIPELINE] Starting: 305 name(s) in 26 batch(es) using enhanced search
[PIPELINE] Batch size: 12, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/26 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/26 =====
[PIPELINE] Names in this batch: ['A. Leon Higginbotham', 'A. Stephen Morse', 'Abraham Tesser', 'Akasha Gloria Hull', 'Akira Suzuki', 'Alan Jay Perlis', 'Albert W. Overhauser', 'Alexandra Boltasseva', 'Alfred E. Mann', 'Alice H. Eagly', 'Alondra Nelson', 'Alvin C. Plantinga']
[BATCH] Processing 12 names: A. Leon Higginbotham, A. Stephen Morse, Abraham Tesser, Akasha Gloria Hull, Akira Suzuki, Alan Jay Perlis, Albert W. Overhauser, Alexandra Boltasseva, Alfred E. Mann, Alice H. Eagly, Alondra Nelson, Alvin C. Plantinga
[BATCH] Phase 1: Running searches in parallel for all 12 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Alvin C. Plantinga: 13 results (mode=basic_only, queries=1, bucket=plausible, t_basic=3.2s, t_rescue=0.0s, t_enh=0.0s, t_total=3.2s)
[PROGRESS] Starting LLM analysis for: Alvin C. Plantinga
[BATCH] Search succeeded for A. Leon Higginbotham: 17 results (mo

Processing batches:   0%|          | 0/1 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/1 =====
[PIPELINE] Names in this batch: ['Arden L. Bement, Jr.', 'Henry L. Roediger III', 'Henry L. Roediger, III', 'Paul Erdos', 'Stephen D. Bechtel', 'Stephen Davison Bechtel']
[BATCH] Processing 6 names: Arden L. Bement, Jr., Henry L. Roediger III, Henry L. Roediger, III, Paul Erdos, Stephen D. Bechtel, Stephen Davison Bechtel
[BATCH] Phase 1: Running searches in parallel for all 6 names (max 16 concurrent)
[BATCH] Phase 2: Streaming LLM as each search completes (max 6 concurrent)
[BATCH] Search succeeded for Henry L. Roediger, III: 14 results (mode=basic_only, queries=1, bucket=plausible, t_basic=7.8s, t_rescue=0.0s, t_enh=0.0s, t_total=7.8s)
[PROGRESS] Starting LLM analysis for: Henry L. Roediger, III
[BATCH] Search succeeded for Paul Erdos: 7 results (mode=basic_only, queries=1, bucket=borderline, t_basic=12.7s, t_rescue=0.0s, t_enh=0.0s, t_total=12.7s)
[PROGRESS] Starting LLM analysis for: Paul Erdos
[PROGRESS] LLM analysis completed for Paul Erdos in 5

e:\Awards DB Code\step3_institution_checker\src\institution_checker\main.py:3040: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_values = merged_values.fillna(False).astype(bool)


In [5]:
# 5. ANALYSIS, OPTIONAL RECOVERY, & EXPORT (FRESH RUN ONLY)
# -----------------------------------------------------------------------------
# Analyze results from this kernel's latest run only.

# Optional: add expected connected names to recover if they were missed.
EXPECTED_CONNECTED_NAMES = []

EXPORT_FILENAME = 'institution_check_results.xlsx'

if 'final_results' not in globals():
    raise RuntimeError("No in-memory results found. Run Section 4 first.")
if not globals().get('RUN_RESULTS_READY', False):
    raise RuntimeError("Results are not marked fresh. Re-run Section 4 in this kernel.")
if 'CURRENT_RUN_ID' not in globals() or CURRENT_RUN_ID is None:
    raise RuntimeError("Missing CURRENT_RUN_ID. Re-run Section 1 then Section 4.")
if '_run_id' not in final_results.columns:
    raise RuntimeError("Result metadata missing (_run_id). Re-run Section 4.")

run_ids = set(final_results['_run_id'].dropna().astype(int).unique().tolist())
if run_ids != {int(CURRENT_RUN_ID)}:
    raise RuntimeError(
        f"Stale/mixed results detected: expected run_id={CURRENT_RUN_ID}, found={sorted(run_ids)}. Re-run Section 4."
    )

if EXPECTED_CONNECTED_NAMES:
    expected_df = final_results[final_results['name'].isin(EXPECTED_CONNECTED_NAMES)].copy()
    pre_missing = expected_df[expected_df['purdue_connection'] != True]['name'].dropna().astype(str).tolist()
    print(f"Expected connected names provided: {len(EXPECTED_CONNECTED_NAMES)}")
    print(f"Missing before recovery: {len(pre_missing)}")

    if pre_missing:
        recovered = await run_targeted_recovery_pass(pre_missing, max_concurrency=4, debug=False)
        recovered_map = {str(r.get('name', '')).strip(): r for r in recovered if isinstance(r, dict)}

        updatable_fields = [
            'institution', 'verdict', 'relationship_type', 'relationship_timeframe',
            'verification_detail', 'summary', 'primary_source', 'verification_status',
            'temporal_context', 'confidence', 'pre_llm_bucket', 'pre_llm_stage',
            'pre_llm_score', 'pre_llm_summary', 'pre_llm_reason_codes', 'pre_llm_used_rescue_query'
        ]

        applied = 0
        for idx, row in final_results.iterrows():
            row_name = str(row.get('name', '')).strip()
            rec = recovered_map.get(row_name)
            if not rec:
                continue

            verdict = str(rec.get('verdict', '')).lower()
            institution_text = str(rec.get('institution', '')).lower()
            summary_text = str(rec.get('summary', '')).lower()
            detail_text = str(rec.get('verification_detail', '')).lower()
            connected_to_purdue = (
                verdict == 'connected'
                and ('purdue' in institution_text or 'purdue' in summary_text or 'purdue' in detail_text)
            )

            if not connected_to_purdue:
                continue

            for f in updatable_fields:
                if f in final_results.columns and f in rec:
                    final_results.at[idx, f] = rec.get(f)
            if 'purdue_connection' in final_results.columns:
                final_results.at[idx, 'purdue_connection'] = True
            applied += 1

        print(f"Recovery overrides applied: {applied}")

print(f"RESULTS ANALYSIS (run_id={CURRENT_RUN_ID})")
print("=" * 70)

total_records = len(final_results)
purdue_records = final_results[final_results['purdue_connection'] == True]
percentage = (len(purdue_records) / total_records * 100) if total_records > 0 else 0

print(f"   Total records: {total_records:,}")
print(f"   Purdue-affiliated: {len(purdue_records):,} ({percentage:.1f}%)")

RESULTS ANALYSIS (run_id=1775669329)
   Total records: 607
   Purdue-affiliated: 424 (69.9%)


In [ ]:
print(f"Run ID: {CURRENT_RUN_ID}")
print(f"Total records: {total_records:,}")
print(f"Purdue-affiliated records: {len(purdue_records):,} ({percentage:.2f}%)")

final_results.to_excel("E:/Ace/Purdue Full Analysis 4.xlsx", index=False)

Run ID: 1775669329
Total records: 607
Purdue-affiliated records: 424 (69.85%)
